# Track A — Signal Selection & Model Training

**Skeleton only. Write the code yourself.**

Assumes you have done `data-analysis.ipynb`. Questions 11–34 in `TODO_RESEARCH.md`
map onto the sections below, and Part 6 of that file lists the decisions already
hardcoded in `src/` that you should reproduce here.

Main tool: `src/research/signal_eval.py` — it has `information_coefficient`,
`ic_summary`, `fama_macbeth`, `spread_portfolio`, `compare_signals`. Read it before
using it.

### Setup

Imports and load the panel. Build the signals with
`src.utils.signals.build_signals(panel)` and look at what keys come back.

## 1. The three candidates

Read `src/utils/signals.py`. For each of the three signals, write down the economic
story — why *should* it predict returns?

**Q12:** why does `momentum_12_1` skip the most recent 21 days?

**Q11/Q14:** one of the three was included as a falsification control, not as a
candidate. Which, and why would you include a signal you expect to fail?

## 2. No look-ahead

Run `assert_causal(panel)`. It rebuilds each signal on truncated panels and demands
an exact match against the full-sample row.

Then convince yourself it actually works: write a deliberately cheating signal (one
that uses `.shift(-1)`), register it in `SIGNAL_LIBRARY`, and check the guard catches
it. `src/research/validate.py` section 5 does exactly this if you want a reference.

## 3. Does the ranking predict returns?

Compute the Information Coefficient for all three signals at horizons 1, 5 and 21.
`compare_signals` will do all of it at once, or build it up yourself from
`information_coefficient` + `ic_summary` if you want to see the mechanics.

**Q13–Q14:** which signals are significant? Is that what you expected?

Now the Fama-MacBeth regression.

**Q15:** does it agree with the IC? Why bother running both — what does the
regression give you that the correlation does not?

**Q16:** forward returns here are open-to-open from t+1. Find the line in
`backtester.py::_run_track_a` that makes that the correct convention. What would you
be measuring if you used close-to-close instead?

**Q17:** compute one naive t-stat and one Newey-West t-stat on the same h=21 IC
series. How big is the gap, and why does overlapping data cause it?

## 4. Try to break it

**Q18:** split into sub-periods and recompute mean IC in each. Is one period carrying
the whole result?

**Q19:** the sample contains Feb–Mar 2020 — a crash and rebound, which is itself an
enormous reversal event. Recompute excluding roughly Feb–Jun 2020.

If the signal only works because of those four months, it is not a strategy.

**Q20–Q21:** sweep the reversal lookback over 5 / 10 / 21 / 42 / 63 / 126 days and
plot IC t-stat against lookback.

Is it a plateau or a single spike? If 21 is not the argmax, why does the code use 21?

**Q22–Q23:** compute the cross-sectional rank correlation between the three signals,
and the correlation of each against trailing volatility.

Two of the three are nearly the same thing. Which two — and what does that tell you
about where the volatility adjustment has to happen?

## 5. From signal to portfolio

**Q24:** build an equal-weight top-6 / bottom-6 spread portfolio (`spread_portfolio`).
Is its Sharpe significant? If the IC was significant but this is not, where did the
edge go?

**Q25:** re-run holding for 5, 21 and 63 days. Relate what you see to the horizon
where the IC was strongest. This is the reasoning behind `rebalance_cadence` in
`config.yaml` — see hint (b) in `TODO_RESEARCH.md`.

## 6. Volatility model

**Q26:** plot all four estimators from `models/volatility.py` for one ticker. Which
responds fastest? Which has a "plateau then cliff" shape, and why?

**Q27–Q28:** why EWMA rather than a fitted GARCH per asset? Why is Parkinson's range
estimator more efficient than close-to-close, and what does it systematically miss?

Write the answers down — this is the "volatility modelling justification" deliverable.

### Calibration

Run the real backtester via `ParticipantStrategy`, then compare
`strat.diagnostics_frame()['ex_ante_vol'].mean()` against realised volatility
(`cagr_pct / sharpe_ratio`).

If the forecast matches realised, say so — a calibrated risk model is the strongest
single piece of evidence for this deliverable.

## 7. Sizing rule

**Q29–Q31:** rank the four rules in `models/optimizer.py` by how much they trust the
covariance matrix. Predict which is most fragile out of sample, then run
`sweep.py --axis sizing` and see whether the backtest agrees.

## 8. Sensitivity

**Q32–Q33:** run the full sweep (`python -m src.research.sweep`), load
`results/sensitivity.csv`, and plot the Sharpe distribution across all configurations.

What fraction are positive? Several settings beat the baseline — should you adopt the
best one? Compare the gaps to the width of the Sharpe confidence interval.

**Q34:** run `sweep.py --axis costs`. How much Sharpe per basis point of slippage?

`config.yaml` sets slippage to 0 but `BacktestConfig` defaults to 5bp. Which will the
graders use?

### The rejected configuration

Run `sweep.py --axis neutrality`. The asymmetric 8-long / 4-short book scores the best
Sharpe of anything in the grid — and it was rejected. See hint (g) in
`TODO_RESEARCH.md`, then go back to your answer to Q5.

Decide for yourself whether that rejection was right. If you disagree, change the code
and be ready to defend it.

## 9. Walk-forward — not yet implemented

**Q35–Q36.** This is a required deliverable and nothing in the repo does it. Every
number above is single-split in-sample.

Design it here: training window length, step size, and — given the strategy has no
fitted parameters — work out what is actually being re-estimated in each fold.